In [ ]:
# Ô 1 — Cài thư viện + nối Google Drive (chạy lại mỗi lần mở Colab)
import os, sys, importlib
try:
    import google.colab  # chỉ có khi chạy trên Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install tldextract rapidfuzz shap xgboost
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/AIC211_URL'   # thư mục chung của nhóm
else:
    ROOT = os.path.abspath('AIC211_URL')          # chạy thử trên máy (không phải Colab)

for d in ['data/raw', 'data/processed', 'src', 'models', 'results', 'figures', 'notebooks']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
sys.path.insert(0, f'{ROOT}/src')
print('ROOT =', ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# Ô 2 — Nạp common.py (của Trí), attacks.py (tuần 4), dữ liệu A và model
# Sửa nhỏ common.py (tuần 5): log_result ghi được các thí nghiệm có cột khác nhau. Chạy nhiều lần không sao.
PATCH = """
# --- v2 (tuan 5): log_result chiu duoc cac thi nghiem co cot khac nhau
def log_result(root, exp, model, source, metrics, defense='none', seed=0, k=0, ops='-', mode='-'):
    path = f'{root}/results/results.csv'
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'), 'exp': exp, 'model': model,
           'defense': defense, 'source': source, 'seed': seed, 'k': k, 'ops': ops, 'mode': mode, **metrics}
    new = pd.DataFrame([row])
    if os.path.exists(path):
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
"""
_cp = f'{ROOT}/src/common.py'
if '# --- v2' not in open(_cp, encoding='utf-8').read():
    with open(_cp, 'a', encoding='utf-8') as f:
        f.write(PATCH)
import common, attacks; importlib.reload(common); importlib.reload(attacks)
from common import *
from attacks import *
import json, joblib, random

assert os.path.exists(f'{ROOT}/data/processed/A.parquet'), 'Chưa có A.parquet: chờ Trí chạy notebook tuần 4.'
assert os.path.exists(f'{ROOT}/src/attacks.py'), 'Chưa có src/attacks.py: chạy lại notebook TV3 tuần 4 trước.'
A = pd.read_parquet(f'{ROOT}/data/processed/A.parquet')
benign_domains = set(A[A.label == 0]['reg_domain'])

thr_path = f'{ROOT}/models/thresholds.json'
if os.path.exists(f'{ROOT}/models/RF_F1.joblib') and os.path.exists(thr_path):
    rf_f1 = joblib.load(f'{ROOT}/models/RF_F1.joblib')
    THR = json.load(open(thr_path))
    MODEL_SOURCE = 'model của Trí (ngưỡng đã đóng băng)'
else:   # Trí chưa xong -> tự train RF-F1 TẠM để viết code trước. Số liệu KHÔNG dùng cho báo cáo.
    from sklearn.ensemble import RandomForestClassifier
    tr, va = A.split == 'train', A.split == 'val'
    rf_f1 = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1,
                                   class_weight='balanced', random_state=42).fit(make_F1(A[tr]), A.label[tr])
    THR = {'RF_F1': threshold_at_fpr(A.label[va], get_scores(rf_f1, make_F1(A[va])))}
    MODEL_SOURCE = 'RF-F1 TẠM (tự train) — chạy lại khi Trí có thresholds.json'
print('Model:', MODEL_SOURCE, '| ngưỡng RF_F1 =', round(THR['RF_F1'], 4))

In [ ]:
# Ô 3 — Ghi file src/e3.py: tấn công ngẫu nhiên, tấn công tham lam hộp đen, score_fn, ASR
E3_CODE = r"""
import random
import numpy as np, pandas as pd
from common import normalize_url, split_parts, get_scores, make_F1
from attacks import apply_op, is_valid

FEATURIZE = {'F1': make_F1, 'F2': lambda d: d[['host', 'rest']]}

def make_score_fn(model, featurize, preprocess=None):
    # Hộp đen: nhận list URL -> mảng điểm độc. preprocess = phòng thủ D1 (None = không phòng thủ)
    def score_fn(urls):
        u = [preprocess(x) if preprocess else x for x in urls]
        d = pd.DataFrame({'url_norm': [normalize_url(x) for x in u]})
        parts = d['url_norm'].map(split_parts)
        d['host'] = parts.str[0]; d['rest'] = parts.str[1]
        return get_scores(model, featurize(d))
    return score_fn

def random_attack(url, k, ops, rng, benign_domains):
    # Kẻ tấn công không biết gì: chọn bừa k biến đổi hợp lệ
    cur, applied, tries = url, [], 0
    while len(applied) < k and tries < 20 * max(k, 1):
        tries += 1
        op = rng.choice(ops)
        new = apply_op(op, cur, rng)
        if is_valid(cur, new, op, benign_domains):
            cur, applied = new, applied + [op]
    return cur, applied

def greedy_attack(url, score_fn, thr, k, ops, rng, benign_domains, n_cand=3):
    # Hộp đen có truy vấn: mỗi bước thử mọi loại biến đổi (n_cand cách mỗi loại), giữ cách làm điểm giảm nhiều nhất
    cur = url; s = score_fn([cur])[0]; queries = 1; applied = []
    for _ in range(k):
        if s <= thr: break                       # đã lọt -> dừng
        cands = []
        for op in ops:
            for _ in range(n_cand):
                new = apply_op(op, cur, rng)
                if is_valid(cur, new, op, benign_domains):
                    cands.append((op, new))
        if not cands: break
        sc = score_fn([c[1] for c in cands]); queries += len(cands)
        j = int(np.argmin(sc))
        if sc[j] >= s: break                     # không giảm được nữa
        cur, s = cands[j][1], sc[j]; applied.append(cands[j][0])
    return {'adv_url': cur, 'ops': applied, 'queries': queries}

def attack_success_rate(mal_urls, score_fn, thr, k, ops, mode, benign_domains, seed=0):
    # ASR = trong số URL độc BAN ĐẦU bị bắt, bao nhiêu % lọt sau khi biến đổi
    rng = random.Random(seed)
    base = score_fn(mal_urls)
    detected = [u for u, s in zip(mal_urls, base) if s > thr]
    rows = []
    for u in detected:
        if mode == 'random':
            adv, applied = random_attack(u, k, ops, rng, benign_domains)
            rows.append({'adv_url': adv, 'ops': applied, 'queries': 0})
        else:
            rows.append(greedy_attack(u, score_fn, thr, k, ops, rng, benign_domains))
    res = pd.DataFrame(rows)
    if len(res):
        res['evaded'] = score_fn(list(res['adv_url'])) <= thr
    asr = float(res['evaded'].mean()) if len(res) else 0.0
    n_evaded = int(res['evaded'].sum()) if len(res) else 0
    return {'asr': asr,
            'recall_clean': len(detected) / max(len(mal_urls), 1),
            'recall_attack': (len(detected) - n_evaded) / max(len(mal_urls), 1),
            'n_detected': len(detected),
            'mean_queries': float(res['queries'].mean()) if len(res) else 0.0}, res
"""
with open(f'{ROOT}/src/e3.py', 'w', encoding='utf-8') as f:
    f.write(E3_CODE)
import e3; importlib.reload(e3)
from e3 import *
score_rf = make_score_fn(rf_f1, FEATURIZE['F1'])
print('Đã ghi', f'{ROOT}/src/e3.py')

In [ ]:
# Ô 4 — Kiểm tra score_fn: recall sạch trên URL độc của A-test phải gần với số của Trí
te = A[(A.split == 'test')]
mal_test = [to_attack_form(u) for u in te[te.label == 1]['url_norm']]
s = score_rf(mal_test)
print(f'Recall sạch RF-F1 trên A-test: {(s > THR["RF_F1"]).mean():.3f}  (so với bảng E2 của Trí)')

# Phát hiện nhỏ: normalize_url bỏ http/https, nên T4 KHÔNG đổi được điểm (ghi vào báo cáo)
u = mal_test[0]
print('Điểm gốc:', round(float(score_rf([u])[0]), 4), '| sau T4:', round(float(score_rf([apply_op("T4", u, random.Random(0))])[0]), 4))

In [ ]:
# Ô 5 — E3 BẢN THỬ: RF-F1, 200 URL độc, k ∈ {1, 3}, ngẫu nhiên vs tham lam, miễn phí vs tất cả
rng = random.Random(0)
trial = random.Random(1).sample(mal_test, min(200, len(mal_test)))
OPS = {'free': FREE_OPS, 'all': FREE_OPS + PAID_OPS}
rows = []
for mode in ['random', 'greedy']:
    for ops_name, ops in OPS.items():
        for k in [1, 3]:
            m, res = attack_success_rate(trial, score_rf, THR['RF_F1'], k, ops, mode, benign_domains, seed=0)
            log_result(ROOT, 'E3_trial', 'RF_F1', 'A-test', m, seed=0, k=k, ops=ops_name, mode=mode)
            rows.append({'kiểu': mode, 'biến đổi': ops_name, 'k': k, **m})
E3_trial = pd.DataFrame(rows)
display(E3_trial.round(3))
print('Kỳ vọng: ASR tham lam ≥ ngẫu nhiên; "all" ≥ "free"; k = 3 ≥ k = 1. Ngược lại thì kiểm tra code.')

In [ ]:
# Ô 6 — Xem vài URL né thành công bằng tấn công tham lam (dạng vô hiệu hóa)
m, res = attack_success_rate(trial[:60], score_rf, THR['RF_F1'], 3, FREE_OPS + PAID_OPS, 'greedy', benign_domains, seed=0)
for _, r in res[res.evaded].head(5).iterrows():
    print(r['ops'], f"| {r['queries']} truy vấn |", defang(r['adv_url']))

In [ ]:
# Ô 7 — Bắt đầu D1: chuẩn hóa đầu vào → src/defenses.py
DEF_CODE = r"""
import re, unicodedata
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from attacks import HOMOGLYPH

CONFUSABLE = {v: k for k, v in HOMOGLYPH.items()}   # bản rút gọn; báo cáo ghi rõ là bảng nhỏ
TRACKING = re.compile(r'^(utm_\w+|ref|fbclid|gclid|mc_cid|mc_eid)$')

def d1_normalize(url):
    # Giải punycode -> Unicode -> đổi ký tự giả (Cyrillic) về Latin; bỏ tham số tracking
    p = urlsplit(url if '://' in url else 'http://' + url)
    labels = []
    for lab in (p.hostname or '').split('.'):
        if lab.startswith('xn--'):
            try: lab = lab.encode('ascii').decode('idna')
            except UnicodeError: pass
        lab = unicodedata.normalize('NFKC', lab)
        labels.append(''.join(CONFUSABLE.get(ch, ch) for ch in lab))
    q = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True) if not TRACKING.match(k.lower())]
    return urlunsplit((p.scheme, '.'.join(labels), p.path, urlencode(q), ''))
"""
with open(f'{ROOT}/src/defenses.py', 'w', encoding='utf-8') as f:
    f.write(DEF_CODE)
import defenses; importlib.reload(defenses)
from defenses import d1_normalize

u = to_attack_form('account-blog16.top/login/verify.php?id=7')
for op in ['T1', 'T7']:
    adv = apply_op(op, u, random.Random(1))
    print(op, '|', adv, '\n   D1 ->', d1_normalize(adv))

In [ ]:
# Ô 8 — D1 cứu được bao nhiêu? (thử nhanh: chỉ bật D1 lúc chấm điểm, CHƯA train lại)
score_rf_d1 = make_score_fn(rf_f1, FEATURIZE['F1'], preprocess=d1_normalize)
rows = []
for op in ['T1', 'T7']:
    m0, _ = attack_success_rate(trial, score_rf,    THR['RF_F1'], 1, [op], 'random', benign_domains, seed=0)
    m1, _ = attack_success_rate(trial, score_rf_d1, THR['RF_F1'], 1, [op], 'random', benign_domains, seed=0)
    rows.append({'T': op, 'ASR không phòng thủ': m0['asr'], 'ASR có D1': m1['asr']})
display(pd.DataFrame(rows).round(3))
print('Lưu ý: đây chỉ là thử nhanh. Tuần 6 phải train lại model trên URL đã qua d1_normalize rồi mới đo chính thức.')